In [12]:
# ============================================================
# dask_pipeline.ipynb
# Dask ML Pipeline - NYC Taxi Dataset
# ============================================================

# 1. Imports

import time

import requests
import dask.dataframe as dd
import numpy as np
import pandas as pd

from pydantic import ByteSize
from dask.distributed import Client
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBRegressor

from dask_ml.model_selection import train_test_split
from dask_ml.linear_model import LogisticRegression as DaskLogisticRegression
from xgboost.dask import DaskXGBRegressor

In [13]:
# 2. Start Dask client

client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 12,Total memory: 27.66 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:59959,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:59994,Total threads: 3
Dashboard: http://127.0.0.1:59995/status,Memory: 6.91 GiB
Nanny: tcp://127.0.0.1:59963,


2026-06-02 20:55:54,672 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c6850ee7eed947e38c00c9ecb2cd1cd3 initialized by task ('shuffle-transfer-c6850ee7eed947e38c00c9ecb2cd1cd3', 0) executed on worker tcp://127.0.0.1:59988
2026-06-02 20:55:54,692 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c6850ee7eed947e38c00c9ecb2cd1cd3 deactivated due to stimulus 'task-finished-1780430154.691549'
2026-06-02 20:55:57,033 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 721cebcc20ca8b8fcbe5924928df9a5b initialized by task ('shuffle-transfer-721cebcc20ca8b8fcbe5924928df9a5b', 0) executed on worker tcp://127.0.0.1:59988
2026-06-02 20:55:57,284 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 721cebcc20ca8b8fcbe5924928df9a5b deactivated due to stimulus 'task-finished-1780430157.283418'
2026-06-02 20:55:59,347 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2287df7957770efc3bf4dec68257579c initialized by task ('shuffle-transfer-2287df7957770efc

In [14]:
# 3. Configuration

RUN_MODE = "local"

if RUN_MODE == "local":
    DATA_PATH = "C:/Users/Tubias/Big Data/Assignment 2/yellow_tripdata_2026-01.parquet"
else:
    DATA_PATH = "gs://your-bucket/taxi/yellow_tripdata_2026-01.parquet"

LOOKUP_PATH = "C:/Users/Tubias/Big Data/Assignment 2/taxi_zone_lookup.csv"

SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"

TARGET = "fare_amount"

FEATURES = [
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "payment_type",
]

SEED = 42
TEST_SIZE = 0.2

LOGISTIC_MAX_ITER = 500
LOGISTIC_REGULARIZATION = 0.0

XGB_N_ESTIMATORS = 100
XGB_MAX_DEPTH = 6

XGB_LEARNING_RATE = 0.1

In [15]:
from collections.abc import Generator
from contextlib import contextmanager
from datetime import timedelta
from pathlib import Path
from time import perf_counter
from typing import TypedDict

class Duration(TypedDict):
    duration: timedelta


def _mem_str(path: str) -> str:
    path = Path(path)
    size = path.stat().st_size
    return ByteSize(size).human_readable(decimal=True)

@contextmanager
def timeit(name: str) -> Generator[dict]:
    result = {"duration": timedelta()}
    start_time = perf_counter()
    yield result
    end_time = perf_counter()
    duration = timedelta(seconds=end_time - start_time)
    print(f"{name} took {duration}")
    result["duration"] = duration

In [16]:
from datetime import timedelta

# 4. Load data
df: dd.DataFrame = None
try:
    with timeit("Loading data"):
        df = dd.read_parquet(DATA_PATH)
except FileNotFoundError:
    response = requests.get(SOURCE_URL)
    response.raise_for_status()
    Path(DATA_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(DATA_PATH, "wb") as f:
        f.write(response.content)
    print(f"Downloaded {DATA_PATH}, {_mem_str(DATA_PATH)}")


if df is None:
    with timeit("Loading data"):
        df = dd.read_parquet(DATA_PATH)

location_lookup = dd.read_csv("C:/Users/Tubias/Big Data/Assignment 2/taxi_zone_lookup.csv")

df.head()

Loading data took 0:00:00.005485


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [17]:
# 5. Basic dataset inspection

print("Columns:")
print(df.columns)

print("Dtypes:")
print(df.dtypes)

print("Number of partitions:")
print(df.npartitions)

Columns:
Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')
Dtypes:
VendorID                           int32
tpep_pickup_datetime      datetime64[us]
tpep_dropoff_datetime     datetime64[us]
passenger_count                    int64
trip_distance                    float64
RatecodeID                         int64
store_and_fwd_flag       string[pyarrow]
PULocationID                       int32
DOLocationID                       int32
payment_type                       int64
fare_amount                      float64
extra                            float64
mta_tax                          float64
tip_amount                       floa

In [18]:
# 6. Preprocessing

df = df[FEATURES + [TARGET]]
df = df.dropna()
df = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["passenger_count"] > 0)
]
df.tail()

,trip_distance,passenger_count,PULocationID,DOLocationID,payment_type,fare_amount
2636822,2.75,2.0,166,237,1,17.0
2636827,1.86,1.0,230,239,1,12.1
2636828,2.66,1.0,239,262,1,16.3
2636829,0.36,4.0,170,170,1,5.8
2636830,1.15,1.0,230,229,1,10.7


In [ ]:
#Acho que quase tudo até eu voltar a comentar mudou
def benchmark_dask(data_path, lookup_path, scenario="Standard"):
    """
    Scenarios: 'Standard', 'Filtered', 'Filtered_Cached'
    """
    import numpy as np
    import dask.dataframe as dd
    metrics = {}
    
    # 1. READ FILE / INITIAL SETUP
    
    with timeit("read file") as duration:
        if scenario == "Standard":
            df = dd.read_parquet(data_path)
            df_lookup = dd.read_csv(lookup_path)
        elif scenario == "Filtered":
            df = dd.read_parquet(data_path)
            df = df[(df['tip_amount'] >= 1) & (df['tip_amount'] <= 5)]
            df_lookup = dd.read_csv(lookup_path)
        elif scenario == "Filtered_Cached":
            df = dd.read_parquet(data_path)
            df = df[(df['tip_amount'] >= 1) & (df['tip_amount'] <= 5)]
            df = df.persist() 
            df_lookup = dd.read_csv(lookup_path).compute() # Broadcast optimization
            
    metrics['read file'] = duration["duration"]

    # Map the NYC taxi series to match generic Databricks terminology
    sa = df['trip_distance']
    sb = df['fare_amount']
    sb_safe = sb.where(sb > 0, 1.0) # Avoid division by zero in raw Standard math

    # 2. STANDARD BENCHMARKS
    
    with timeit("count") as duration:
        _ = len(df)
    metrics['count'] = duration["duration"]
    
    with timeit("count index") as duration:
        _ = len(df.index)
    metrics['count index'] = duration["duration"]
    
    with timeit("mean") as duration:
        _ = sa.mean().compute()
    metrics['mean'] = duration["duration"]
    
    with timeit("standard deviation") as duration:
        _ = sa.std().compute()
    metrics['standard deviation'] = duration["duration"]
    
    with timeit("value counts") as duration:
        _ = sa.value_counts().compute()
    metrics['value counts'] = duration["duration"]

    # 3. SERIES MATH & ARITHMETIC
    
    with timeit("series addition") as duration:
        res_add = sa + sb
    metrics['series addition'] = duration["duration"]
    
    with timeit("mean of series addition") as duration:
        _ = res_add.mean().compute()
    metrics['mean of series addition'] = duration["duration"]
    
    with timeit("series multiplication") as duration:
        res_mul = sa * sb
    metrics['series multiplication'] = duration["duration"]
    
    with timeit("mean of series multiplication") as duration:
        _ = res_mul.mean().compute()
    metrics['mean of series multiplication'] = duration["duration"]

    with timeit("complex arithmetic") as duration:
        res_complex = np.sin(sa) * np.cos(sb_safe) + np.arctan2(sa, sb_safe)
    metrics['complex arithmetic'] = duration["duration"]
    
    with timeit("mean of complex arithmetic") as duration:
        _ = res_complex.mean().compute()
    metrics['mean of complex arithmetic'] = duration["duration"]

    # 4. AGGREGATIONS & JOINS
    
    with timeit("groupby statistics") as duration:
        _ = df.groupby('passenger_count')['trip_distance'].agg(['mean', 'std']).compute()
    metrics['groupby statistics'] = duration["duration"]
    
    # Clean data types outside the timer to keep the join graph pure
    df_lookup['LocationID'] = df_lookup['LocationID'].astype(np.int32)
    
    with timeit("join") as duration:
        merged_df = df.merge(df_lookup, left_on='PULocationID', right_on='LocationID')
    metrics['join'] = duration["duration"]
    
    with timeit("join count") as duration:
        _ = len(merged_df)
    metrics['join count'] = duration["duration"]
    
    return metrics

In [ ]:
benchmarks_standard = benchmark_dask(DATA_PATH, LOOKUP_PATH, "Standard")

read file took 0:00:00.007238
count took 0:00:00.007410
count index took 0:00:00.171409
mean took 0:00:00.157933
standard deviation took 0:00:00.106167
value counts took 0:00:00.155014
series addition took 0:00:00.000295
mean of series addition took 0:00:00.128315
series multiplication took 0:00:00.000332
mean of series multiplication took 0:00:00.111853
complex arithmetic took 0:00:00.001557
mean of complex arithmetic took 0:00:00.212024
groupby statistics took 0:00:00.161573
join took 0:00:00.005041
join count took 0:00:00.942282


In [ ]:
benchmarks_filtered = benchmark_dask(DATA_PATH, LOOKUP_PATH, "Filtered")

read file took 0:00:00.009359
count took 0:00:00.129119
count index took 0:00:00.091713
mean took 0:00:00.148045
standard deviation took 0:00:00.168448
value counts took 0:00:00.445531
series addition took 0:00:00.000399
mean of series addition took 0:00:00.202927
series multiplication took 0:00:00.000328
mean of series multiplication took 0:00:00.178151
complex arithmetic took 0:00:00.002045
mean of complex arithmetic took 0:00:00.231312
groupby statistics took 0:00:00.207603
join took 0:00:00.004731
join count took 0:00:00.207089


In [ ]:
benchmarks_cached = benchmark_dask(DATA_PATH, LOOKUP_PATH, "Filtered_Cached")

read file took 0:00:00.025784
count took 0:00:00.805420
count index took 0:00:00.025562
mean took 0:00:00.037024
standard deviation took 0:00:00.051458
value counts took 0:00:00.895080
series addition took 0:00:00.000921
mean of series addition took 0:00:00.038641
series multiplication took 0:00:00.000526
mean of series multiplication took 0:00:00.041007
complex arithmetic took 0:00:00.001933
mean of complex arithmetic took 0:00:00.080318
groupby statistics took 0:00:00.069212
join took 0:00:00.007913
join count took 0:00:00.098837


In [23]:
from pathlib import Path

yellow2025: dd.DataFrame = None

yellow2025 = dd.read_parquet("C:/Users/Tubias/Big Data/Assignment 2/yellow_tripdata_*.parquet") #Tirei o Loop dado que é uma tarefa linear enquanto que o nosso objetivo é optimizar processos em paralelo - isto faz a mesma coisa
yellow2025 = yellow2025.repartition(npartitions=12) # Isto divide os 3 meses em 12 partes iguais para usar os 12 threads do meu pc - !!!Tem de ser alterado para o numero de threads do pc

yellow2025.to_parquet("C:/Users/Tubias/Big Data/Assignment 2/2025_yellow_tripdata.parquet", write_index=False)

In [ ]:
benchmarks_standard = benchmark_dask("C:/Users/Tubias/Big Data/Assignment 2/2025_yellow_tripdata.parquet", LOOKUP_PATH, "Standard")

read file took 0:00:00.018383
count took 0:00:00.032395
count index took 0:00:00.048793
mean took 0:00:00.137326
standard deviation took 0:00:00.193530
value counts took 0:00:00.390866
series addition took 0:00:00.000329
mean of series addition took 0:00:00.172873
series multiplication took 0:00:00.000370
mean of series multiplication took 0:00:00.191565
complex arithmetic took 0:00:00.001690
mean of complex arithmetic took 0:00:00.299288
groupby statistics took 0:00:00.253485
join took 0:00:00.005804
join count took 0:00:00.312547


In [ ]:
benchmarks_filtered = benchmark_dask("C:/Users/Tubias/Big Data/Assignment 2/2025_yellow_tripdata.parquet", LOOKUP_PATH, "Filtered")

read file took 0:00:00.013224
count took 0:00:00.148336
count index took 0:00:00.167013
mean took 0:00:00.234753
standard deviation took 0:00:00.226588
value counts took 0:00:00.267054
series addition took 0:00:00.000346
mean of series addition took 0:00:00.340465
series multiplication took 0:00:00.000382
mean of series multiplication took 0:00:00.304040
complex arithmetic took 0:00:00.001764
mean of complex arithmetic took 0:00:00.318689
groupby statistics took 0:00:00.367257
join took 0:00:00.005207
join count took 0:00:00.306389


In [ ]:
benchmarks_cached = benchmark_dask("C:/Users/Tubias/Big Data/Assignment 2/2025_yellow_tripdata.parquet", LOOKUP_PATH, "Filtered_Cached")

read file took 0:00:00.636477
count took 0:00:00.051548
count index took 0:00:00.039644
mean took 0:00:00.113441
standard deviation took 0:00:00.059962
value counts took 0:00:00.227089
series addition took 0:00:00.000706
mean of series addition took 0:00:00.069300
series multiplication took 0:00:00.000618
mean of series multiplication took 0:00:00.062396
complex arithmetic took 0:00:00.002507
mean of complex arithmetic took 0:00:00.131313
groupby statistics took 0:00:00.088181
join took 0:00:00.006705
join count took 0:00:00.113859


In [ ]:
#Nada mudou abaixo
X = df[FEATURES]
y_reg = df[TARGET]

In [28]:
X_train, X_test, y_train_reg, y_test_reg = train_test_split(

    X,
    y_reg,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True

)

reg_model = DaskXGBRegressor(

    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    objective="reg:squarederror",
    random_state=SEED

)

with timeit("Regression training") as duration:
    fitted_reg_model = reg_model.fit(
        X_train,
        y_train_reg
    )

regression_train_time = duration["duration"]

with timeit("Regression prediction") as duration:
    y_pred_reg = fitted_reg_model.predict(X_test).compute()

regression_prediction_time = duration["duration"]

y_test_reg_local = y_test_reg.compute()

Regression training took 0:00:05.130111
Regression prediction took 0:00:00.822310


In [29]:
regression_results = {
    "library": "Dask",
    "task": "regression",
    "model": "DaskXGBRegressor",
    "train_time": regression_train_time,
    "prediction_time": regression_prediction_time,
    "mae": mean_absolute_error(y_test_reg_local, y_pred_reg),
    "mse": mean_squared_error(y_test_reg_local, y_pred_reg),
    "rmse": mean_squared_error(y_test_reg_local, y_pred_reg) ** 0.5,
    "r2": r2_score(y_test_reg_local, y_pred_reg),
}

In [30]:
def classify_fare_partition(partition):
    return partition.assign(
        fare_class=np.select(
            [
                partition[TARGET] < 15,
                partition[TARGET] < 40,
            ],
            [0, 1],
            default=2
        )
    )


df_cls = df.map_partitions(classify_fare_partition)

X_cls = df_cls[FEATURES]
y_cls = df_cls["fare_class"]

X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True
)

X_train_arr = X_train.to_dask_array(lengths=True)
X_test_arr = X_test.to_dask_array(lengths=True)

y_train_arr = y_train_cls.to_dask_array(lengths=True)
y_test_arr = y_test_cls.to_dask_array(lengths=True)

cls_model = DaskLogisticRegression(
    max_iter=LOGISTIC_MAX_ITER,
    fit_intercept=True
)

with timeit("Classification training") as duration:
    fitted_cls_model = cls_model.fit(X_train_arr, y_train_arr)

classification_train_time = duration["duration"]

with timeit("Classification prediction") as duration:
    y_pred_cls = fitted_cls_model.predict(X_test_arr).compute()

classification_prediction_time = duration["duration"]

y_test_cls_local = y_test_arr.compute()

Classification training took 0:00:14.217514
Classification prediction took 0:00:00.612325


In [31]:
classification_results = {
    "library": "Dask",
    "task": "classification",
    "model": "DaskLogisticRegression",
    "train_time": classification_train_time,
    "prediction_time": classification_prediction_time,
    "accuracy": accuracy_score(y_test_cls_local, y_pred_cls),
    "precision_macro": precision_score(y_test_cls_local, y_pred_cls, average="macro"),
    "recall_macro": recall_score(y_test_cls_local, y_pred_cls, average="macro"),
    "f1_macro": f1_score(y_test_cls_local, y_pred_cls, average="macro"),
}

c:\Users\Tubias\.conda\envs\bdcc-env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [32]:
#Save Results 

benchmark_results_df = pd.DataFrame([
    {
        "library": "Dask",
        "operation": operation,
        "duration": duration,
        "duration_seconds": duration.total_seconds()
    }

    for operation, duration in benchmarks.items()
])

benchmark_results_df.to_csv("C:/Users/Tubias/Big Data/Assignment 2/results/dask_benchmark_results.csv",index=False)

results_df = pd.DataFrame([
    regression_results,
    classification_results
])

results_df.to_csv("C:/Users/Tubias/Big Data/Assignment 2/results/dask_pipeline_results.csv", index=False)

results_df

,library,task,model,train_time,prediction_time,mae,mse,rmse,r2,accuracy,precision_macro,recall_macro,f1_macro
0,Dask,regression,DaskXGBRegressor,0 days 00:00:05.130111,0 days 00:00:00.822310,2.782677,38.175703,6.178649,0.886141,NaN,NaN,NaN,NaN
1,Dask,classification,DaskLogisticRegression,0 days 00:00:14.217514,0 days 00:00:00.612325,NaN,NaN,NaN,NaN,0.768858,0.496506,0.567697,0.525652
